# 7a Machine Learning - Part I

In this part of the exercise you will take a processed DataFrame containing all provided spectral data, divide it into training and test subsets, and use these to set up and test a k-nearest neighbours machine learning model.
<br>
<br>Once trained, this model should be able to classify unknown substitution patterns based on the labelled spectral data it has been presented with.

---

**Note:** Running this code will prevent some irrelevant errors popping up later on in the exercise.

In [12]:
import warnings
warnings.filterwarnings("ignore")

---

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.axes3d as p3
%matplotlib inline
import C317
import os
os.scandir("/content/drive/MyDrive/Raw_IR_Spectra")
import sklearn.decomposition

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import importlib
importlib.reload(C317)

a = C317.load_spectra(20)
a.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
    o-salicylaldehyde_4.txt  o-1-chloro-2-nitrobenzene_3.txt  \
0                 -0.000849                        -0.000447   
1                 -0.000827                        -0.000170   
2                  0.000180                        -0.000037   
3                 -0.000098                         0.000216   
4                  0.000221                         0.000097   
5                  0.000331                        -0.000463   
6                 -0.000040                        -0.000401   
7                 -0.000173                        -0.000320   
8                  0.000206                         0.000248   
9                  0.000183                        -0.000332   
10                 0.000043                        -0.000127   
11                -0.000053                         0.000103   
12                 0.000097            

,o-salicylaldehyde,o-1-chloro-2-nitrobenzene,o-salicylic acid,o-guaiacol,o-aminobenzylalcohol,m-tolualdehyde,o-bromotoluene,m-toluoyl chloride,p-acetoxybenzoic,m-aminoacetophenone,...,p-(trifluoromethyl)benzaldehyde,p-aminophenol,p-hydroxybenzoic acid,p-tolualdehyde,p-iodoanisole,p-iodoanisole,p-iodophenol,p-(trifluoromethyl)benzaldehyde,p-iodoanisole,p-acetamidophenol
630.0,0.000186,0.000212,0.000170,0.000241,0.000148,0.000266,0.000294,0.000250,0.000233,0.000209,...,0.000293,0.000246,0.000222,0.000298,0.000236,0.000236,0.000153,0.000289,0.000235,0.000183
631.0,0.000184,0.000211,0.000166,0.000240,0.000147,0.000264,0.000291,0.000249,0.000230,0.000209,...,0.000294,0.000245,0.000222,0.000297,0.000235,0.000234,0.000162,0.000292,0.000236,0.000185
632.0,0.000178,0.000209,0.000168,0.000240,0.000147,0.000262,0.000291,0.000246,0.000228,0.000207,...,0.000295,0.000246,0.000221,0.000297,0.000236,0.000233,0.000164,0.000294,0.000236,0.000186
633.0,0.000173,0.000207,0.000164,0.000242,0.000147,0.000259,0.000291,0.000243,0.000225,0.000206,...,0.000296,0.000246,0.000219,0.000296,0.000236,0.000234,0.000145,0.000295,0.000235,0.000187
634.0,0.000171,0.000204,0.000162,0.000241,0.000146,0.000256,0.000291,0.000240,0.000226,0.000204,...,0.000295,0.000247,0.000216,0.000296,0.000235,0.000235,0.000143,0.000298,0.000234,0.000185


---

### Train/Test Splits

When testing the success of a machine learning algorithm, a dataset is typically divided into two, a training dataset and a test dataset.
<br>
<br>The training dataset is the information that is passed to the algorithm and what it bases its 'learning' on. While the test dataset is used as a benchmark to see how well the algorithm can predict the right result (correctly identify data subsets) after learning from the training set.
<br>
<br> When assigning train-test splits, i.e. dividing the data into the two subsets, it is important that datasets containing repeated measurements are treated so that all repeats are passed into the same one of the two sets. Consider a case in which four of five repeated measurements are passed to the training dataset and the remaining one is passed to the test dataset. Since the algorithm will have encountered four instances of the same data already within the training dataset, it is highly likely that it will succesfully assign the final repeat in the test dataset. As a result, the apparent success of the machine learning is overexaggerated and not a true representation of the success of the machine learning.
<br>
<br>Thus, while it is useful to save spectra with numbers indicating different repeats (computers do not like files with exactly the same name), it is now best to rename the data so that these repeat indicators are removed.

In [4]:
column_names = []
for i in os.scandir("/content/drive/MyDrive/Raw_IR_Spectra"):
  column_names.append(i.name)
print(column_names)

['o-salicylaldehyde_4.txt', 'o-1-chloro-2-nitrobenzene_3.txt', 'o-salicylic acid_4.txt', 'o-guaiacol_3.txt', 'o-aminobenzylalcohol_3.txt', 'm-tolualdehyde_3.txt', 'o-bromotoluene_3.txt', 'm-toluoyl chloride_3.txt', 'p-acetoxybenzoic_2.txt', 'm-aminoacetophenone_1.txt', 'o-anthranilicacid_4.txt', 'm-tolualdehyde_4.txt', 'o-aspirin_5.txt', 'o-dicyanobenzene_3.txt', 'o-ethoxyphenol_5.txt', 'p-allylphenoxide_2.txt', 'o-fluorobenzaldehyde_5.txt', 'o-chlorobenzaldehyde_3.txt', 'o-toluidine_1.txt', 'm-diethoxybenzene_5.txt', 'm-toluenesulfonyl chloride_4.txt', 'p-hydroxybenzamide_1.txt', 'm-methyl o-toluate_5.txt', 'p-bromotoluene_3.txt', 'o-aminobenzylamine_3.txt', 'm-acetoxybenzoic_4.txt', 'm-nitrotoluene_3.txt', 'm-aminoacetophenone_4.txt', 'o-hydroxybenzonitrile_3.txt', 'o-guaiacol_5.txt', 'p-allylphenoxide_1.txt', 'o-fluorobenzaldehyde_2.txt', 'o-toluidine_4.txt', 'o-methylacetophenone_2.txt', 'o-dicyanobenzene_5.txt', 'p-bromoacetophenone_4.txt', 'p-cyanobenzaldehyde_1.txt', 'm-methyl o

In [5]:
# re.split(pattern, string, maxsplit=0, flags=0)
# re.split(r'\W+', 'Words, words, words.')
import re
columns_identical = []
for i in column_names:
  columns_identical.append(re.split(r'\_+', i)[0])

print(columns_identical)

['o-salicylaldehyde', 'o-1-chloro-2-nitrobenzene', 'o-salicylic acid', 'o-guaiacol', 'o-aminobenzylalcohol', 'm-tolualdehyde', 'o-bromotoluene', 'm-toluoyl chloride', 'p-acetoxybenzoic', 'm-aminoacetophenone', 'o-anthranilicacid', 'm-tolualdehyde', 'o-aspirin', 'o-dicyanobenzene', 'o-ethoxyphenol', 'p-allylphenoxide', 'o-fluorobenzaldehyde', 'o-chlorobenzaldehyde', 'o-toluidine', 'm-diethoxybenzene', 'm-toluenesulfonyl chloride', 'p-hydroxybenzamide', 'm-methyl o-toluate', 'p-bromotoluene', 'o-aminobenzylamine', 'm-acetoxybenzoic', 'm-nitrotoluene', 'm-aminoacetophenone', 'o-hydroxybenzonitrile', 'o-guaiacol', 'p-allylphenoxide', 'o-fluorobenzaldehyde', 'o-toluidine', 'o-methylacetophenone', 'o-dicyanobenzene', 'p-bromoacetophenone', 'p-cyanobenzaldehyde', 'm-methyl o-toluate', 'o-1-chloro-2-nitrobenzene', 'p-dimethoxybenzene', 'm-tolualdehyde', 'm-anisaldehyde', 'p-bromotoluene', 'm-diacetoxybenzene', 'm-bromobenzaldehyde', 'm-toluenesulfonyl chloride', 'o-hydroxybenzonitrile', 'o-chl

Recall that the column titles of a DataFrame can be overwritten by just setting the `.columns` attribute of the DataFrame to the new list of column titles.

In [6]:
import importlib
importlib.reload(C317)

a = C317.load_spectra(20)
a.columns = columns_identical # I didn't know this way of using .columns attribute
a.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
    o-salicylaldehyde_4.txt  o-1-chloro-2-nitrobenzene_3.txt  \
0                 -0.000849                        -0.000447   
1                 -0.000827                        -0.000170   
2                  0.000180                        -0.000037   
3                 -0.000098                         0.000216   
4                  0.000221                         0.000097   
5                  0.000331                        -0.000463   
6                 -0.000040                        -0.000401   
7                 -0.000173                        -0.000320   
8                  0.000206                         0.000248   
9                  0.000183                        -0.000332   
10                 0.000043                        -0.000127   
11                -0.000053                         0.000103   
12                 0.000097            

,o-salicylaldehyde,o-1-chloro-2-nitrobenzene,o-salicylic acid,o-guaiacol,o-aminobenzylalcohol,m-tolualdehyde,o-bromotoluene,m-toluoyl chloride,p-acetoxybenzoic,m-aminoacetophenone,...,p-(trifluoromethyl)benzaldehyde,p-aminophenol,p-hydroxybenzoic acid,p-tolualdehyde,p-iodoanisole,p-iodoanisole,p-iodophenol,p-(trifluoromethyl)benzaldehyde,p-iodoanisole,p-acetamidophenol
630.0,0.000186,0.000212,0.000170,0.000241,0.000148,0.000266,0.000294,0.000250,0.000233,0.000209,...,0.000293,0.000246,0.000222,0.000298,0.000236,0.000236,0.000153,0.000289,0.000235,0.000183
631.0,0.000184,0.000211,0.000166,0.000240,0.000147,0.000264,0.000291,0.000249,0.000230,0.000209,...,0.000294,0.000245,0.000222,0.000297,0.000235,0.000234,0.000162,0.000292,0.000236,0.000185
632.0,0.000178,0.000209,0.000168,0.000240,0.000147,0.000262,0.000291,0.000246,0.000228,0.000207,...,0.000295,0.000246,0.000221,0.000297,0.000236,0.000233,0.000164,0.000294,0.000236,0.000186
633.0,0.000173,0.000207,0.000164,0.000242,0.000147,0.000259,0.000291,0.000243,0.000225,0.000206,...,0.000296,0.000246,0.000219,0.000296,0.000236,0.000234,0.000145,0.000295,0.000235,0.000187
634.0,0.000171,0.000204,0.000162,0.000241,0.000146,0.000256,0.000291,0.000240,0.000226,0.000204,...,0.000295,0.000247,0.000216,0.000296,0.000235,0.000235,0.000143,0.000298,0.000234,0.000185


In [7]:
# df[df.columns[0]]
# a[a.columns[1]]
# a.columns[1]

# df.loc["Melting point"]

a2 = a[["m-toluicacid"]].copy()
print(a2)

       m-toluicacid  m-toluicacid  m-toluicacid  m-toluicacid  m-toluicacid
630.0      0.000258      0.000254      0.000260      0.000250      0.000248
631.0      0.000258      0.000254      0.000261      0.000251      0.000250
632.0      0.000258      0.000255      0.000262      0.000249      0.000247
633.0      0.000256      0.000253      0.000261      0.000249      0.000252
634.0      0.000257      0.000254      0.000261      0.000249      0.000249
...             ...           ...           ...           ...           ...
876.0      0.000209      0.000205      0.000223      0.000201      0.000198
877.0      0.000206      0.000203      0.000222      0.000199      0.000195
878.0      0.000205      0.000199      0.000218      0.000197      0.000193
879.0      0.000203      0.000199      0.000217      0.000194      0.000190
880.0      0.000199      0.000197      0.000217      0.000192      0.000188

[251 rows x 5 columns]


Notice that now all five repeats stay together, which is what we wanted.

In [3]:
import importlib
import C317
importlib.reload(C317)

a = C317.load_spectra(20)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
    o-salicylaldehyde  o-1-chloro-2-nitrobenzene  o-salicylic acid  \
0           -0.000849                  -0.000447         -0.000650   
1           -0.000827                  -0.000170         -0.000150   
2            0.000180                  -0.000037          0.000207   
3           -0.000098                   0.000216         -0.000342   
4            0.000221                   0.000097          0.000236   
5            0.000331                  -0.000463          0.000031   
6           -0.000040                  -0.000401          0.000258   
7           -0.000173                  -0.000320          0.000061   
8            0.000206                   0.000248         -0.000035   
9            0.000183                  -0.000332         -0.000028   
10           0.000043                  -0.000127         -0.000068   
11          -0.000053          

In [6]:
b = set(a.columns)
b = list(b)
print(b)

['p-acetoxybenzoic', 'o-methylacetophenone', 'm-dimethoxybenzene', 'm-nitrotoluene', 'p-trans-anethole', 'o-aminobenzylamine', 'p-cyanobenzaldehyde', 'p-allylphenoxide', 'p-bromobenzoic acid', 'm-acetoxybenzoic', 'm-aminoacetophenone', 'p-hydroxybenzamide', 'm-hydroxybenzonitrile', 'm-methoxybenzaldehyde', 'm-toluoyl chloride', 'o-salicylic acid', 'p-(trifluoromethyl)acetophenone', 'o-aspirin', 'p-iodophenol', 'p-cyanophenol', 'm-bromobenzaldehyde', 'o-chlorobenzaldehyde', 'o-fluorophenol', 'p-xylene', 'p-fluoroaniline', 'p-acetaminophen', 'm-tolualdehyde', 'm-anisaldehyde', 'm-toluenesulfonyl chloride', 'p-chlorobenzaldehyde', 'p-hydroxybenzaldehyde', 'p-fluoroacetophenone', 'o-anthranilicacid', 'o-aminobenzylalcohol', 'm-toluicacid', 'o-dichlorobenzene', 'o-fluorobenzaldehyde', 'm-diethoxybenzene', 'o-bromotoluene', 'p-bromobenzaldehyde', 'p-bromotoluene', 'p-aminobenzoic', 'p-toluidine', 'm-diacetylbenzene', 'o-guaiacol', 'p-dimethoxybenzene', 'o-hydroxybenzonitrile', 'm-diacetoxybe

The library `sklearn.model_selection` has a function `train_test_split()`.
<br>
<br>`train_test_split` takes in a list, and splits it into two - a training dataset and a test dataset. By default the splitting size is 75:25 training:test, but this can be altered by passing a different value of the test_size parameter (`test_size=x`, where `x` is a value between 0 and 1 for 0% of the full dataset to 100% making up the test dataset).
<br>
<br>`train_samples, test_samples = sklearn.model_selection.train_test_split(samples,test_size=x)`
<br>
<br>**Note:** Each time `train_test_split()` is employed it will return a different distribution of data between the two subsets.

In [18]:
import sklearn.model_selection
train_samples, test_samples = sklearn.model_selection.train_test_split(b, test_size = 0.33)

In [19]:
training = a[train_samples]
testing = a[test_samples]
print(training)
print(testing)

       m-diethoxybenzene  m-diethoxybenzene  m-diethoxybenzene  \
630.0           0.000291           0.000290           0.000290   
631.0           0.000291           0.000292           0.000290   
632.0           0.000291           0.000290           0.000291   
633.0           0.000291           0.000292           0.000290   
634.0           0.000292           0.000291           0.000291   
...                  ...                ...                ...   
876.0           0.000287           0.000283           0.000286   
877.0           0.000287           0.000285           0.000285   
878.0           0.000288           0.000286           0.000288   
879.0           0.000288           0.000290           0.000288   
880.0           0.000288           0.000289           0.000287   

       m-diethoxybenzene  m-diethoxybenzene  p-hydroxybenzamide  \
630.0           0.000290           0.000290            0.000201   
631.0           0.000289           0.000292            0.000199   
632.0 

In [20]:
train_labels = []
test_labels = []
for i in training.columns:
    train_labels.append(i[0])
for i in testing.columns:
    test_labels.append(i[0])
print(train_labels)
print(test_labels)

['m', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'm', 'm', 'm', 'm', 'm', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'm', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'm', 'm', 'm', 'm', 'm', 'm', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'm', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'm', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'm', 'm', 'm', 'm', 'm', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'm', 'm', 'm', 'm', 'm', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'p', 'o', 'o', 'o', 'o', 'o', 'm', 'm', 'm', 'm', 'm', 'o', 'o', 'o', 'o', 'o',

---

### The k-Nearest Neighbours Algorithm

With an appropriately prepared dataset separated into training and testing subsets, it is now possible to pass the data to a machine learning algorithm. In this case you will employ the k-nearest neighbours algorithm, which categorises data based on distance in dataspace.
<br>
<br>We will employ the `sklearn.neighbors` library to carry out the machine learning.

In [21]:
import sklearn.neighbors

The `sklearn.neighbors` library lets you create an object called a `KNeighborsClassifier()`, which takes as an argument the number of nearest neighbours to use. (You might draw parallels between this and the `PCA` object generated in the previous notebook.)

In [22]:
knn = sklearn.neighbors.KNeighborsClassifier(n_neighbors=3)

The `KNeighborsClassifier` object has a method called `fit(training data, training labels)`, which sets up/trains your model, ready to make predictions. Much like with PCA, the `sklearn` library expects your data with samples as rows, and features as columns (as you encountered with PCA). Recall that a DataFrame has an attribute which could help here.

In [23]:
knn.fit(training.T, train_labels)

KNeighborsClassifier(n_neighbors=3)

You have now made a KNearestNeighbours object, and trained it with your processed data.

The `score(testing data, testing labels)` method can be used to test a trained `KNeighborsClassifier` object with testing data. It returns a score of what percentage of test data your model correctly classifies.

**Note:** Again, the testing data must be handed to the method with samples as rows and features as columns.

In [24]:
score = knn.score(testing.T, test_labels)
print(score)

0.7461538461538462


You will likely get a value between 60% and 90%. The variability is caused by different possible test/train splits - if you train your model with a different subset of the data, it will perform differently.

In [41]:
def knn(df, test_size=0.33, n_neighbors=3):
    compounds = list(set(df.columns))
    train_samples, test_samples = sklearn.model_selection.train_test_split(
        compounds, test_size=test_size)
    training = df[train_samples]
    testing = df[test_samples]
    train_labels = [i[0] for i in training.columns] # much more concise than define list -> for loop -> append
    test_labels = [i[0] for i in testing.columns]
    knn = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
    knn.fit(training.T, train_labels)
    score = knn.score(testing.T, test_labels)
    # print(score)
    return score

print(score)

0.7461538461538462


To get a better idea of the accuracy of the model, it is better to average the score over a few hundred different test train splits (recall that each time you use `train_test_split()` it generates a new split).

In [42]:
import numpy as np

scores = []
for i in range(500):
    scores.append(knn(a))

mean_score = np.round(np.mean(scores), 4)
std_error = np.round(np.std(scores) / np.sqrt(len(scores)), 4)
print(f"Mean score: {mean_score}")
print(f"Standard error: {std_error}")

Mean score: 0.7422
Standard error: 0.0033


Once a `KNeighborsClassifier` object has been trained, it is also possible to employ the `predict()` method to attempt to assign a particular subset of data from your test subset.

In [38]:
print(testing.columns)

Index(['o-fluorobenzaldehyde', 'o-fluorobenzaldehyde', 'o-fluorobenzaldehyde',
       'o-fluorobenzaldehyde', 'o-fluorobenzaldehyde', 'm-dimethoxybenzene',
       'm-dimethoxybenzene', 'm-dimethoxybenzene', 'm-dimethoxybenzene',
       'm-dimethoxybenzene',
       ...
       'p-(trifluoromethyl)acetophenone', 'p-(trifluoromethyl)acetophenone',
       'p-(trifluoromethyl)acetophenone', 'p-(trifluoromethyl)acetophenone',
       'p-(trifluoromethyl)acetophenone', 'm-methoxybenzaldehyde',
       'm-methoxybenzaldehyde', 'm-methoxybenzaldehyde',
       'm-methoxybenzaldehyde', 'm-methoxybenzaldehyde'],
      dtype='object', length=130)


In [44]:
knn_model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=3)
knn_model.fit(training.T, train_labels)

# Then predict:
predictions = knn_model.predict(testing['o-fluorobenzaldehyde'].T)
print("o-fluorobenzaldehyde predictions:", predictions)

predictions2 = knn_model.predict(testing['m-dimethoxybenzene'].T)
print("m-dimethoxybenzene predictions:", predictions2)

o-fluorobenzaldehyde predictions: ['o' 'o' 'o' 'o' 'o']
m-dimethoxybenzene predictions: ['m' 'm' 'm' 'm' 'm']


---